# Thuật toán Lượng tử Tìm Bậc (Quantum Order-Finding) cho $N = 15$
**Mục tiêu:** Cho số nguyên $x$ nguyên tố cùng nhau với $N = 15$ ($\gcd(x, 15) = 1$), tìm số nguyên dương nhỏ nhất $r$ sao cho:
$$x^r \equiv 1 \pmod{15}$$

Quy trình gồm **3 mảnh ghép**:
1. **Mảnh 1 (Quantum Gate $U_x$):** Thiết kế cổng $U_x\vert y\rangle = \vert xy \bmod 15\rangle$ trên 4 qubit.
2. **Mảnh 2 (QPE & Sampling):** Đưa $U_x$ vào hàm `build_qpe`, lấy mẫu bằng `StatevectorSampler` (Qiskit Primitives V2) để thu các đỉnh pha $\varphi \approx \frac{s}{r}$.
3. **Mảnh 3 (Continued Fractions):** Khai triển liên phân số từ $\varphi$ để bóc ra mẫu số $r$.

In [11]:
%pip install -q qiskit qiskit-aer pylatexenc

In [30]:
import time
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Gate
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator

## 0. Tái sử dụng khối `build_inverse_qft`, `build_qpe` và `gcd`

In [31]:
def build_inverse_qft(n_qubits: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits, name="Inverse_QFT")
    for i in range(n_qubits - 1, -1, -1):
        for j in range(n_qubits - 1, i, -1):
            qc.cp(-(2 * np.pi) / (2 ** (j - i + 1)), j, i)
        qc.h(i)
    for i in range(n_qubits // 2):
        qc.swap(i, n_qubits - 1 - i)
    return qc


def build_qpe(t_qubits: int, unitary_gate: Gate, state_prep_circ: QuantumCircuit | None = None) -> QuantumCircuit:
    qr = QuantumRegister(t_qubits, name="counting")
    num_target = unitary_gate.num_qubits
    qtarget = QuantumRegister(num_target, name="target")
    qc = QuantumCircuit(qr, qtarget, name="QPE")

    if state_prep_circ is not None:
        qc.compose(state_prep_circ, qubits=qtarget, front=False, inplace=True)
    else:
        qc.x(qtarget[0])  # Khởi tạo |0001> = |1>

    qc.h(qr)

    curr_unitary_gate = unitary_gate
    for i in range(t_qubits):
        ctrl_gate = curr_unitary_gate.control()
        qc.append(ctrl_gate, [qr[i], *qtarget])
        curr_unitary_gate = curr_unitary_gate.power(2)

    inv_qft = build_inverse_qft(t_qubits)
    qc.compose(inv_qft, qubits=qr, inplace=True, front=False)
    return qc


def gcd(a: int, b: int) -> int:
    if b == 0:
        return abs(a)
    return gcd(b, a % b)

## 1. Mảnh 1: Thiết kế cổng nhân mô-đun $U_x\vert y\rangle = \vert xy \bmod N\rangle$ bằng Ma trận Hoán vị (`UnitaryGate`)

Với $N$ bất kỳ ($L = \lceil \log_2 N \rceil$ qubit $\implies$ ma trận $2^L \times 2^L$):

$t = 2 \times L + 1 + \lceil \log_2(2 + 1 \div (2 \times \epsilon))\rceil$
* Với mọi trạng thái $0 \le y < N$: cột $y$ đặt phần tử `1` tại hàng $y' = (x \cdot y) \bmod N$ để thực hiện ánh xạ $\vert y\rangle \to \vert xy \bmod N\rangle$.
* Với các trạng thái dư $N \le y < 2^L$ : giữ nguyên $\vert y\rangle \to \vert y\rangle$ (đặt `1` trên đường chéo chính $U_{y, y} = 1$) để ma trận đảm bảo tính Unitary ($U^\dagger U = I$).

In [32]:
def build_mod_N_gate (x : int, N : int , epsilon : float = 0.25) -> Gate:
    if (gcd(x, N) != 1): raise(ValueError(f"{x} và {N} không phải co-prime"))
    
    # Khởi tạo L(target_qbits), t(counting_qbits_)
    target_qbits = int(np.ceil(np.log2(N)))
    counting_qbits = 2 * target_qbits + 1 + np.ceil(np.log2(2 +  1 / (2 * epsilon)))
    
    # Tạo ma trận vuông 2**L x 2**L (số chều trong không gian Hilbert)
    dim = 2 ** target_qbits
    U = np.zeros((dim, dim), dtype=complex)
    for i in range(dim):
        if i < N:
            U[(x * i) % N, i] = 1
        else:
            U[i, i] = 1
    return UnitaryGate(U, label=f"U_{x}mod{N}") # trả ma trận về dạng Gate

## 2. Mảnh 2: Chạy mạch QPE và đo pha $\varphi$ (Chuẩn Qiskit Primitives V2)

Ta nối cổng `U_gate = build_mod_N_gate(x, N, epsilon)` vào hàm `build_qpe(t_qubits, U_gate)`, chỉ gắn phép đo vào thanh ghi `counting` (`counting_qubits` đầu tiên), và chạy với `StatevectorSampler`.

In [36]:
def run_qpe_for_order(x: int, N: int = 15, epsilon : float = 0.25, counting_qubits: int = 4, shots: int = 1024) -> dict[str, int]:
    U_gate = build_mod_N_gate(x, N, epsilon) # tạo cổng U
    
    qc = build_qpe(t_qubits= counting_qubits, unitary_gate= U_gate) # tạo QPE
    
    cr = ClassicalRegister(counting_qubits)
    # Measure Phase
    aer_simulator = AerSimulator()
    
    qc.add_register(cr)
    # chỉ measure target qubits (vì trong hàm build_qpe target qubits được tạo sau counting qubits nên chạy ở cuối array)
    qc.measure(qc.qubits[:counting_qubits], cr) 
    
    # chạy simulator
    job = aer_simulator.run(transpile(qc, aer_simulator), shots= shots)
    result = job.result()
    
    counts = result.get_counts() # trả về dict
    return counts

## 3. Mảnh 3: Thuật toán Khai triển Liên phân số (Continued Fractions) để tìm bậc $r$

### Mục đích
Từ chuỗi bit đo được trên thanh ghi `counting`, ta có phân số pha chính xác:
$$\varphi = \frac{\text{num}}{\text{den}} = \frac{\text{int}(\text{bitstring}, 2)}{2^t} \approx \frac{s}{r}$$
Thuật toán Liên phân số chuyển $\frac{\text{num}}{\text{den}}$ thành dãy các **phân số hội tụ (convergents)** $\frac{p_k}{q_k}$ ($q_k < N$) để tìm lại mẫu số $r$ thật (kể cả khi $\varphi$ bị lệch nhẹ do rò rỉ pha).

### Các bước hoạt động của `continued_fraction_convergents(num, den, max_den=N)`
1. **Tách phần nguyên & lật ngược phần dư (giống thuật toán Euclid):**
   $$\frac{a}{b} = \underbrace{(a \mathbin{//} b)}_{\text{coeff } a_k} + \cfrac{1}{\;\dfrac{b}{a \bmod b}\;}$$
   Sau mỗi bước, lấy `coeff = a // b` rồi cập nhật **`a, b = b, a % b`**.
2. **Gộp các `coeff` thành phân số hội tụ $\frac{p_k}{q_k}$ bằng công thức truy hồi:**
   * Bản thân `coeff` chỉ là hệ số tầng (ví dụ $\frac{12}{16} = \frac{3}{4}$ có `coeff = [0, 1, 3]`, không chứa số `4`), nên ta phải ghép chúng lại bằng:
     $$p_k = a_k \cdot p_{k-1} + p_{k-2}, \qquad q_k = a_k \cdot q_{k-1} + q_{k-2}$$
   * Khởi tạo điều kiện biên ở $k = -2, -1$: **`p_prev2, p_prev1 = 0, 1`** và **`q_prev2, q_prev1 = 1, 0`** để bước đầu tự ra $\frac{a_0}{1}$ và bước hai ra $\frac{a_1 a_0 + 1}{a_1}$.
3. **Hai điều kiện dừng vòng lặp:**
   * **`b == 0`:** Phân số đã chia hết sạch phần dư (ví dụ $\frac{12}{16} \to (16,12) \to (12,4) \to (4,0)$ dừng với dãy hội tụ `0/1, 1/1, 3/4`).
   * **`q_curr >= N`:** Dừng ngay vì bậc $r$ luôn thỏa $r < N$; các phân số có mẫu số $\ge N$ chỉ sinh ra do sai số làm tròn của $2^t$.
4. **Kiểm tra ứng viên bậc $r$:** Duyệt các mẫu số hội tụ $q_k$, kiểm tra xem $x^{q_k} \bmod N == 1$ hay không để chọn ra bậc $r$ nhỏ nhất thỏa mãn.

In [38]:
def continued_fraction_convergents(num: int, den: int, max_den: int) -> list[tuple[int, int]]:
    """
    Tự cài đặt thuật toán Khai triển continued_faction cho phân số num / den.
    Trả về danh sách các phân số hội tụ (p_m, q_m) với q_m < max_den (N).
    """
    convergents = []
    p_prev2, p_prev1 = 0, 1
    q_prev2, q_prev1 = 1, 0

    a, b = num, den
    while b != 0: 
        coeff = a // b
        a, b = b, a % b
        
        p_curr = coeff * p_prev1 + p_prev2
        q_curr = coeff * q_prev1 + q_prev2
        
        if q_curr >= max_den: break # vì r < N nên khi mẫu số >= N thì dừng
        
        convergents.append((p_curr, q_curr))
        
        p_prev2, p_prev1 = p_prev1, p_curr
        q_prev2, q_prev1 = q_prev1, q_curr
    
    return convergents

def find_order_from_counts(x: int, N: int, counts: dict[str, int], t_qubits: int) -> int | None:
    """Phân tích các trạng thái đo được để bóc tách chu kỳ (bậc) r nhỏ nhất."""
    total_shots = sum(counts.values())
    detected_orders = []

    print(f"{'Bitstring':<10} | {'P(x)':<9} | {'Phase (φ)':<10} | {'Hội tụ (s/r)':<14} | {'Kết quả kiểm tra'}")
    print("-" * 78)

    # Sắp xếp theo xác suất giảm dần và lọc P(x) > 1e-4
    for bitstr, cnt in sorted(counts.items(), key=lambda item: item[1], reverse=True):
        prob = cnt / total_shots
        if prob <= 1e-4:
            continue

        measured_int = int(bitstr, 2)
        phase = measured_int / (2 ** t_qubits)

        if measured_int == 0:
            print(f"{bitstr:<10} | {prob:<9.5f} | {phase:<10.5f} | {'0/1':<14} | Bỏ qua (đỉnh tầm thường s = 0)")
            continue

        convergents = continued_fraction_convergents(measured_int, 2 ** t_qubits, max_den=N)
        conv_str = ", ".join(f"{p}/{q}" for p, q in convergents)

        found_r = None
        for p_m, q_m in convergents:
            if q_m > 0 and pow(x, q_m, N) == 1:
                found_r = q_m
                detected_orders.append(q_m)
                break

        if found_r is not None:
            status = f"Thành công -> Tìm thấy bậc r = {found_r} (vì {x}^{found_r} mod {N} = 1)"
        else:
            last_q = convergents[-1][1] if convergents else "?"
            status = f"Chưa đủ (r ứng viên = {last_q}, nhưng {x}^{last_q} mod {N} != 1)"

        print(f"{bitstr:<10} | {prob:<9.5f} | {phase:<10.5f} | {conv_str:<14} | {status}")

    return min(detected_orders) if detected_orders else None

## 4. Chạy thử nghiệm End-to-End Tìm Bậc cho x và N bất kỳ

In [51]:
N = 21
x = 2
target_qbits = int(np.ceil(np.log2(N)))
epsilon = 0.25
counting_qubits = int(2 * target_qbits + 1 + np.ceil(np.log2(2 +  1 / (2 * epsilon))))

counts = run_qpe_for_order(x=x, N=N, counting_qubits=counting_qubits, shots=1024, epsilon=epsilon)
r = find_order_from_counts(x=x, N=N, counts=counts, t_qubits=counting_qubits)
print(f"\n=> KẾT LUẬN: Bậc của x = {x} modulo {N} là r = {r}")

Bitstring  | P(x)      | Phase (φ)  | Hội tụ (s/r)   | Kết quả kiểm tra
------------------------------------------------------------------------------
1000000000000 | 0.17969   | 0.50000    | 0/1, 1/2       | Chưa đủ (r ứng viên = 2, nhưng 2^2 mod 21 != 1)
0000000000000 | 0.17090   | 0.00000    | 0/1            | Bỏ qua (đỉnh tầm thường s = 0)
0010101010101 | 0.11719   | 0.16663    | 0/1, 1/6       | Thành công -> Tìm thấy bậc r = 6 (vì 2^6 mod 21 = 1)
1010101010101 | 0.11133   | 0.66663    | 0/1, 1/1, 1/2, 2/3 | Chưa đủ (r ứng viên = 3, nhưng 2^3 mod 21 != 1)
0101010101011 | 0.10742   | 0.33337    | 0/1, 1/2, 1/3  | Chưa đủ (r ứng viên = 3, nhưng 2^3 mod 21 != 1)
1101010101011 | 0.10254   | 0.83337    | 0/1, 1/1, 5/6  | Thành công -> Tìm thấy bậc r = 6 (vì 2^6 mod 21 = 1)
1101010101010 | 0.03418   | 0.83325    | 0/1, 1/1, 4/5, 5/6 | Thành công -> Tìm thấy bậc r = 6 (vì 2^6 mod 21 = 1)
1010101010110 | 0.02637   | 0.66675    | 0/1, 1/1, 2/3  | Chưa đủ (r ứng viên = 3, nhưng 2^3 mod 21 !